# odor_space_sampling functionality
This is a notebook that explores some of the functionality of this repo. Each section (vaugely) explores the different functions in the scripts of src/odor_space_sampling.

## data.py
This script is predominantly used for loading and working with raw csv files and coverting them into useable odor spaces. For example, it has functions for loading a csv.

In [1]:
from odor_space_sampling import data

path_to_data = "../data/gslf_and_human_data.csv"

df = data.load_csv(path_to_data)
print(df.head())

                smiles     label    cid                    IUPAC
0      Cc1ccnc2ccccc12  ['gslf']  10285        4-methylquinoline
1  CC(C)c1ccnc2ccccc12  ['gslf']  74004     4-isopropylquinoline
2      Cc1cnc2ccccc2n1  ['gslf']  23686      2-methylquinoxaline
3     Cc1nc2ccccc2nc1C  ['gslf']  16925  2,3-dimethylquinoxaline
4      Cc1cccc2nccnc12  ['gslf']  61670      5-methylquinoxaline


And there is a function to convert any csv into rdkit descriptors (as long as the csv has a "smiles" column with smiles strings)

In [2]:
data_matrix = data.make_rdkit_descriptors(path_to_data)
print(f"outputs a {data_matrix.shape} matrix corresponding to {data_matrix.shape[0]} odors by {data_matrix.shape[1]} descriptors")

generating descriptors: 100%|██████████| 5841/5841 [00:15<00:00, 367.08it/s]


outputs a (5841, 217) matrix corresponding to 5841 odors by 217 descriptors


In [3]:
print(data_matrix)

[[ 4.24203704  4.24203704  1.07638889 ...  0.          0.
   0.        ]
 [ 4.32425926  4.32425926  0.56287037 ...  0.          0.
   0.        ]
 [ 4.31314815  4.31314815  0.95657407 ...  0.          0.
   0.        ]
 ...
 [12.3299194  12.3299194   0.58849271 ...  0.          0.
   0.        ]
 [14.83846027 14.83846027  0.02170942 ...  0.          0.
   0.        ]
 [14.35132867 14.35132867  0.         ...  0.          0.
   0.        ]]


Additionally this script can add CID and IUPAC names to the smiles strings in your csv by querying the pubchem api.

In [4]:
# create a csv without any CID or IUPCA names
no_cid_df = df.drop(columns=['cid', 'IUPAC']).iloc[:10]
no_cid_df.to_csv("../data/no_cid_test.csv", index=False) # saving it to a csv
print(no_cid_df)

                   smiles      label
0         Cc1ccnc2ccccc12   ['gslf']
1     CC(C)c1ccnc2ccccc12   ['gslf']
2         Cc1cnc2ccccc2n1   ['gslf']
3        Cc1nc2ccccc2nc1C   ['gslf']
4         Cc1cccc2nccnc12   ['gslf']
5     Cc1nc2ccccc2c(C)c1C  ['human']
6    CC(C)Cc1ccnc2ccccc12   ['gslf']
7  CC(C)(C)c1cccc2ncccc12   ['gslf']
8         Cc1ccc2ccccc2n1   ['gslf']
9         Cc1cccc2ccccc12   ['gslf']


In [5]:
# generate CID and IUPAC names back
cid_df = data.add_cid_to_data(filepath="../data/no_cid_test.csv", save=False)
print("newly generated CIDs and IUPAC names")
print(cid_df)
print("=====")
print("original dataframe")
print(df.iloc[:10])

generating CIDs: 10it [00:04,  2.50it/s]

newly generated CIDs and IUPAC names
                   smiles      label     cid                     IUPAC
0         Cc1ccnc2ccccc12   ['gslf']   10285         4-methylquinoline
1     CC(C)c1ccnc2ccccc12   ['gslf']   74004      4-isopropylquinoline
2         Cc1cnc2ccccc2n1   ['gslf']   23686       2-methylquinoxaline
3        Cc1nc2ccccc2nc1C   ['gslf']   16925   2,3-dimethylquinoxaline
4         Cc1cccc2nccnc12   ['gslf']   61670       5-methylquinoxaline
5     Cc1nc2ccccc2c(C)c1C  ['human']   17096  2,3,4-trimethylquinoline
6    CC(C)Cc1ccnc2ccccc12   ['gslf']   74005       4-isobutylquinoline
7  CC(C)(C)c1cccc2ncccc12   ['gslf']  109127     5-tert-butylquinoline
8         Cc1ccc2ccccc2n1   ['gslf']    7060         2-methylquinoline
9         Cc1cccc2ccccc12   ['gslf']    7002       1-methylnaphthalene
=====
original dataframe
                   smiles      label     cid                     IUPAC
0         Cc1ccnc2ccccc12   ['gslf']   10285         4-methylquinoline
1     CC(C)c1cc

## utils.py
A handful of useful utility functions for working with the data. For example you can do PCA on the rdkit descriptors.

In [6]:
from odor_space_sampling import utils

reduced_data_matrix = utils.reduce_data(data_matrix)

initial data shape: (5841, 217)
dimension after removing constant features: (5841, 193)
dimensionality of 99% explained variance: 107
reduced space shape (5841, 107)


## sampling.py
This script has all of the sampling functions to sample odors in different ways. The easiest way to do all the sampling is to run the function `sample_with_all_methods`.

In [9]:
import pprint
from odor_space_sampling import sampling

all_samples = sampling.sample_with_all_methods(reduced_data_matrix, n_samples=100, seed=12345, n_gaussians=100)
print(all_samples)

{'uniform': {'samples': array([[ -1.16453369,  -7.86071031,  -4.08316609, ...,  -0.65923517,
          1.19426491,   0.32262318],
       [  7.12686789, -10.49183411,  -0.59531238, ...,  -0.28603634,
         -0.56026254,   0.44202601],
       [ 48.42518195, -22.04218417, -23.77979619, ...,  -1.76265909,
         -1.78919027,  -1.24798946],
       ...,
       [ 17.26130461,  15.8991092 ,  -4.54826792, ...,   0.1416046 ,
          0.09845184,   0.05034499],
       [  5.12461287, -11.23569002,  -6.0246075 , ...,   0.23199896,
         -0.44625777,   0.14933913],
       [ -5.52307724,  -5.16903051,  -2.54222852, ...,   0.52002379,
          1.25399945,  -0.90955475]], shape=(100, 107)), 'indices': array([4837, 5529, 5833, 3081, 5839, 5840, 5833, 5783, 5839, 5800, 5782,
       5783, 5839, 5833, 5839, 5813, 3081, 5826, 5831, 5839, 4383, 5065,
       5813, 3100, 5840, 5828, 5822, 5604, 4619, 5604, 5836, 5839, 5833,
       5831, 5839, 5833, 5813, 5826, 5839, 1296, 5833, 5833, 5532, 5428,
     

The output of `sample_with_all_methods` is a nested dictionary with one entry per sampling method:

```python
{
    "sampling_method_1": {
        "samples":   np.ndarray,  # the actual sampled points
        "indices":   np.ndarray,  # indices into the original odor dataset
        "distances": np.ndarray,  # distance from each sample to nearest real odor
    },
    "sampling_method_2": { ... },
    ...
}
```

> **Note:** `"distances"` is only included for methods that don't sample directly from the odor set (e.g. GMM), where sampled points may not correspond to real odors.

## analysis.py
This script is for functions that analyze each sampling method

## plotting.py
This script is for functions that are used in plotting